Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

def load_dataset(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    return df
def load_dataset_tab_separation(file_path):
    df = pd.read_csv(file_path, low_memory=False, sep='\t')
    return df
def load_dataset_encoding(file_path):
    df = pd.read_csv(file_path, low_memory=False, encoding='latin1')
    return df

In [ ]:
# IMDB imports
#df_imdb_name_basics = load_dataset('../data/imdb_datasets/name.basics.csv')   
df_imdb_title_basics = load_dataset('../data/imdb_datasets/title.basics.csv')
#df_imdb_title_akas = load_dataset('../data/imdb_datasets/title.akas.csv')
#df_imdb_title_crew = load_dataset('../data/imdb_datasets/title.crew.csv')
#df_imdb_title_episode = load_dataset('../data/imdb_datasets/title.episode.csv')
#df_imdb_title_principals = load_dataset('../data/imdb_datasets/title.principals.csv')
df_imdb_title_ratings = load_dataset('../data/imdb_datasets/title.ratings.csv') 
# OSCAR import
df_oscar = load_dataset_tab_separation('../data/oscars_1927-2025/full_data.csv')
# BAFTA import
df_bafta = load_dataset('../data/bafta_1949-2020/bafta_films.csv')
# TMDB import
df_tmdb = load_dataset('../data/tmdb_dataset/TMDB_movie_dataset_v11.csv')
# Box Office Mojo import
df_box_office_mojo = load_dataset_encoding('../data/box_office_mojo/box_office_mojo_2015-2025.csv')
# The Numbers import
df_the_numbers = load_dataset_encoding('../data/the_numbers/the_numbers_box_office_2015-2025.csv') 
# Netflix imports
df_netflix_revenue_subs_spend = load_dataset('../data/netflix/netflix_rev_subs_spend.csv')
df_netflix_engagement = load_dataset('../data/netflix/netflix_engagement-report_2023-2025.csv')

In [ ]:
print(len(df_imdb_title_ratings))
print(len(df_tmdb))

In [ ]:
#df_the_numbers = pd.read_csv('../data/clean_data/cleaned_the_numbers.csv', low_memory=False, encoding="utf-8")

df =  df_the_numbers

df['Title'] = (
    df['Title']
    .str.replace(r'[^\w\s:.\-]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print(df['Title'].value_counts().to_string())

EXPLORATORY ANALYSIS

In [ ]:
# SELECT DATAFRAME FOR ANALYSIS

# df of interest:
df =  df_oscar ## START HERE !!

# Print top rows
print(df.head(30).to_string(), flush=True)

# DataFrame Info and Dtypes
#print(df.info())

In [ ]:
# Basic Summary Stats
print(df.describe(include='all')) 

In [ ]:
# Category Analysis / Check

column_name = 'production_companies'
print(df[column_name].value_counts())

#print(df.head(10).to_string(), flush=True)

# BAFTA - clean category column
#df['category clean'] = df['category'].str.extract(r'^Film \|\s*(.*?)\s*in \d{4}$')
#print(df['category clean'].value_counts().to_string()) # this code forces notebooks to not truncate the output of value counts, which is important for this column as there are many unique values.

In [ ]:

matching_rows = df[df['production_companies'].str.lower().str.contains('netflix', na=False)]
print(matching_rows)


DATA CLEANING

In [ ]:
# Drop unwanted / unnecessary columns

# TEMPLATE: df = df.drop(columns=['column1', 'column2', 'column3'])

df = df.drop(columns=['backdrop_path', 'poster_path'])

# Check it worked:
print(df.head(10).to_string(), flush=True)

In [ ]:
# Check Data Types

df.dtypes

# IF APPLICABLE - Convert Data Types
# df['column_name'] = df['column_name'].astype('desired_dtype')

#df['birthYear'] = df['birthYear'].astype('Int64') # Convert to nullable integer type

df['release_date'] = pd.to_datetime(df['release_date'], dayfirst=True, errors='coerce') # Convert to datetime, coercing errors to NaT']
df['release_date'] = df['release_date'].dt.date

df.dtypes
print(df.head(10).to_string(), flush=True)

In [ ]:
print(df.head(10).to_string(), flush=True)

In [ ]:
## String Formatting

# Remove special characters

column_name1 = 'Total Views'

#df[column_name1] = df[column_name1].str.replace('$', '', regex=False)
df[column_name1] = df[column_name1].str.replace(',', '', regex=False)
df[column_name1] = df[column_name1].str.replace('-', '', regex=False)
df[column_name1] = df[column_name1].replace('', np.nan)                # Replace empty strings with NaN so float conversion works
df[column_name1] = df[column_name1].astype('Int64')

#print(df.head(10).to_string(), flush=True)

df.dtypes

In [ ]:
column_name2 = 'foreign%'

df[column_name2] = df[column_name2].str.replace('%', '', regex=False)
df[column_name2] = df[column_name2].str.replace('-', '', regex=False)
df[column_name2] = df[column_name2].str.replace('<', '', regex=False)
df[column_name2] = df[column_name2].replace('', np.nan)                 # Replace empty strings with NaN so float conversion works
df[column_name2] = df[column_name2].astype(float) / 100


In [ ]:
# Netflix - clean columns

col = 'Netflix Subscribers (EMEA)'

df[col] = (
    df[col]
        .astype(str)
        .str.lower()
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.replace('thousand', '*1e3', regex=False)
        .str.replace('million',  '*1e6', regex=False)
        .str.replace('billion',  '*1e9', regex=False)
        .str.replace(' ', '', regex=False)
    )
# convert expressions like "35.89*1e6" into numbers
df[col] = df[col].replace('', np.nan)                  # handle empty cells
df[col] = df[col].map(lambda x: eval(x) if isinstance(x, str) else x)
df[col] = df[col].round().astype('Int64')  # round and convert to integer

In [ ]:
print(df.head(20).to_string(), flush=True)

In [ ]:
# Convert to proper format

column_name = 'title'

df[column_name] = df[column_name].str.lower()  # or .upper()

print(df.head(10).to_string(), flush=True)

In [ ]:
# Check for Missing Values

print("Null values:\n",df.isna().sum())

print("Total number of rows:",df.shape[0])

column_name = 'title'

# Show first 10 rows where column_name is NaN
#null_nomId_rows = df[df[column_name].isna()]
#print(null_nomId_rows.head(10).to_string(index=False))


# IF APPLICABLE - remove or fill missing values:
# TEMPLATE: df.dropna() // df = df.dropna(subset=['column_name']) // df.dropna(inplace=True) // df.fillna(value)

#df = df.dropna(subset=[column_name])

#print("Null values:\n",df.isnull().sum())

In [ ]:
df.dtypes
print(df.head(10).to_string(), flush=True)

In [ ]:
# DUPLICATES

### Check for duplicates - entire row entry
print("Number of duplicate rows:\t", df.duplicated().sum())

# Show duplicated rows
#all_duplicates = df[df.duplicated(keep = False)]
#print("Duplicate rows:\n", all_duplicates)

# Remove duplicates
# df.drop_duplicates()

### Check for duplicates - specific column

#TEMPLATE
column_name = 'title'
duplicates_in_column = df[df[column_name].duplicated(keep=False)]

print("Duplicate rows per column:\n", duplicates_in_column)


In [ ]:
# TMDB title duplicates

#df_tmdb = df_tmdb[df_tmdb['vote_count'] != 0].copy()

dupes = df_tmdb[df_tmdb[column_name].duplicated(keep=False)].copy()
dupes = dupes.sort_values([column_name])  # optional: keep groups together

print(dupes.head(40).sort_values(by='vote_count', ascending=False).to_string(index=False))


In [ ]:
# TMDB example:

df = df.sort_values(
        by='vote_count',      # sort only by vote_count
        ascending=False       # highest vote_count first
    ).drop_duplicates( subset=['id'], keep='first')        # keep the highest-vote_count row

print("Number of duplicate rows:\t", df.duplicated().sum()) # check it worked

print(df[df['title'] == 'the avengers'].to_string(index=False))

In [ ]:
### Check for duplicates - multiple columns

dupes = df[df.duplicated(subset=['title', 'release_date'], keep=False)]
print(len(dupes))
print(dupes.head(10).to_string(index=False))

In [ ]:
# Box Office Mojo example: keep entry with highest worldwide box office for each title + release year combination, and drop the rest as duplicates.
df = (
    df.sort_values(by='worldwide_box_office', ascending=False)
      .drop_duplicates(subset=['title', 'release_year'], keep='first')
      .reset_index(drop=True)
)


In [ ]:
# TMDB example: drop duplicates on title, keeping the one with highest vote count (as above), and drop the rest as duplicates.
df_tmdb = (
    df.sort_values(by='vote_count', ascending=False)
      .drop_duplicates(subset=['title'], keep='first')
      .reset_index(drop=True)
)

In [ ]:
print(df.head(10).to_string(), flush=True)

In [ ]:
# OTHER

column_name = 'Total Views'
# Remove Outliers
sns.boxplot(x=df[column_name])
plt.show()